# 03 -- Data quality: child tables (feature by feature)

Quality checks and cleaning for bureau, previous applications, installments,
POS cash, and credit card tables. Same discipline as notebook 02: inspect,
visualize, clean dirty fields, show before/after.

Large tables are handled **one table at a time** with chunked scans where needed.
Restart the kernel before running if RAM is tight.

Leakage reminder: relative day/month offsets that are after application time
must not feed application-time features (enforced later in SQL + tests).

In [ ]:
from __future__ import annotations

import gc
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DAYS_EMPLOYED_ANOMALY, PROCESSED_DATA_DIR, RAW_DATA_DIR

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
DECISIONS: list[dict] = []
CHUNKSIZE = 500_000
SAMPLE_N = 80_000
DAY_SENTINEL = DAYS_EMPLOYED_ANOMALY  # 365243 used across Home Credit tables

print(PROJECT_ROOT)
print("day sentinel:", DAY_SENTINEL)

In [ ]:
def record(table: str, feature: str, issue: str, action: str, detail: str = "") -> None:
    DECISIONS.append(
        {
            "table": table,
            "feature": feature,
            "issue": issue,
            "action": action,
            "detail": detail,
        }
    )
    print(f"[decision] {table}.{feature}: {action} ({issue})")


def plot_numeric(series: pd.Series, title: str, bins: int = 50, ax=None):
    ax = ax or plt.gca()
    sns.histplot(series.dropna(), bins=bins, kde=False, ax=ax, color="steelblue")
    ax.set_title(title)
    ax.set_xlabel(series.name)
    return ax


def plot_categorical(series: pd.Series, title: str, top_n: int | None = 15, figsize=(8, 4)):
    counts = series.fillna("__MISSING__").value_counts()
    if top_n is not None:
        counts = counts.head(top_n)
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(x=counts.values, y=counts.index.astype(str), ax=ax, color="teal")
    ax.set_title(title)
    ax.set_xlabel("count")
    plt.tight_layout()
    plt.show()
    return counts


def before_after_numeric(before: pd.Series, after: pd.Series, title_b: str, title_a: str, bins: int = 50):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_numeric(before, title_b, bins=bins, ax=axes[0])
    plot_numeric(after, title_a, bins=bins, ax=axes[1])
    plt.tight_layout()
    plt.show()


def feature_report(df: pd.DataFrame, col: str, table: str = "") -> None:
    s = df[col]
    prefix = f"{table}." if table else ""
    print(f"\n=== {prefix}{col} ===")
    print(f"dtype={s.dtype}  missing={s.isna().sum():,} ({100 * s.isna().mean():.2f}%)")
    print(f"n_unique={s.nunique(dropna=True)}")
    if pd.api.types.is_numeric_dtype(s):
        display(s.describe())
    else:
        display(s.value_counts(dropna=False).head(12))


def replace_day_sentinel(s: pd.Series, sentinel: int = DAY_SENTINEL) -> tuple[pd.Series, pd.Series]:
    """Return cleaned series and anomaly flag."""
    flag = (s == sentinel).astype("int8")
    out = s.mask(s == sentinel, np.nan)
    return out, flag


def sample_csv(path: Path, n: int = SAMPLE_N, usecols=None) -> pd.DataFrame:
    """Reservoir-style approx sample via first chunks shuffled (good enough for viz)."""
    parts = []
    taken = 0
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False):
        need = n - taken
        if need <= 0:
            break
        if len(chunk) <= need:
            parts.append(chunk)
            taken += len(chunk)
        else:
            parts.append(chunk.sample(need, random_state=42))
            taken += need
            break
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def chunked_value_counts(path: Path, col: str, top_n: int = 20) -> pd.Series:
    counts: dict[object, int] = {}
    for chunk in pd.read_csv(path, usecols=[col], chunksize=CHUNKSIZE, low_memory=False):
        vc = chunk[col].fillna("__MISSING__").value_counts()
        for k, v in vc.items():
            counts[k] = counts.get(k, 0) + int(v)
        del chunk
    s = pd.Series(counts).sort_values(ascending=False)
    return s.head(top_n)


def chunked_null_and_sentinel(path: Path, col: str, sentinel: int = DAY_SENTINEL) -> dict:
    n = 0
    n_null = 0
    n_sent = 0
    for chunk in pd.read_csv(path, usecols=[col], chunksize=CHUNKSIZE, low_memory=False):
        n += len(chunk)
        n_null += int(chunk[col].isna().sum())
        n_sent += int((chunk[col] == sentinel).sum())
        del chunk
    return {"rows": n, "null": n_null, "sentinel": n_sent}

---
# A. `bureau.csv`

External credit history. One row per bureau credit (`SK_ID_BUREAU`).

In [ ]:
bureau_path = RAW_DATA_DIR / "bureau.csv"
bureau = pd.read_csv(bureau_path, low_memory=False)
print("shape:", bureau.shape)
print("memory_mb:", round(bureau.memory_usage(deep=True).sum() / 1024**2, 1))

miss = (bureau.isna().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=miss.values, y=miss.index, ax=ax, color="salmon")
ax.set_xlabel("% missing")
ax.set_title("bureau.csv missingness")
plt.tight_layout()
plt.show()

## Feature: `SK_ID_CURR` / `SK_ID_BUREAU`

In [ ]:
assert bureau["SK_ID_BUREAU"].isna().sum() == 0
assert bureau["SK_ID_CURR"].isna().sum() == 0
print("duplicate SK_ID_BUREAU:", int(bureau["SK_ID_BUREAU"].duplicated().sum()))
print("unique SK_ID_CURR:", bureau["SK_ID_CURR"].nunique())
record("bureau", "SK_ID_BUREAU", "none", "keep; unique bureau credit id")
record("bureau", "SK_ID_CURR", "none", "keep; join key to application")

## Feature: `CREDIT_ACTIVE`

In [ ]:
feature_report(bureau, "CREDIT_ACTIVE", "bureau")
plot_categorical(bureau["CREDIT_ACTIVE"], "bureau.CREDIT_ACTIVE")
record("bureau", "CREDIT_ACTIVE", "none", "keep as-is")

## Feature: `CREDIT_CURRENCY`

In [ ]:
feature_report(bureau, "CREDIT_CURRENCY", "bureau")
plot_categorical(bureau["CREDIT_CURRENCY"], "bureau.CREDIT_CURRENCY")
record(
    "bureau",
    "CREDIT_CURRENCY",
    "almost all one currency",
    "keep but low modelling value; may drop later",
)

## Feature: `DAYS_CREDIT`

Days before application the credit was opened (should be <= 0).

In [ ]:
feature_report(bureau, "DAYS_CREDIT", "bureau")
print("DAYS_CREDIT > 0:", int((bureau["DAYS_CREDIT"] > 0).sum()))
fig, ax = plt.subplots(figsize=(8, 4))
plot_numeric(bureau["DAYS_CREDIT"], "bureau.DAYS_CREDIT", bins=60, ax=ax)
plt.tight_layout()
plt.show()
record(
    "bureau",
    "DAYS_CREDIT",
    "relative offset",
    "keep; require <= 0 for application-time features",
)

## Feature: `CREDIT_DAY_OVERDUE`

In [ ]:
feature_report(bureau, "CREDIT_DAY_OVERDUE", "bureau")
s = bureau["CREDIT_DAY_OVERDUE"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=(s > 0).astype(int), ax=axes[0], color="salmon")
axes[0].set_title("has overdue days?")
plot_numeric(s[s > 0], "CREDIT_DAY_OVERDUE | overdue only", bins=40, ax=axes[1])
plt.tight_layout()
plt.show()
record("bureau", "CREDIT_DAY_OVERDUE", "sparse positive overdue", "keep as-is")

## Feature: `DAYS_CREDIT_ENDDATE`

Can be positive (planned end after application). Not a dirty spike; document carefully for leakage.

In [ ]:
feature_report(bureau, "DAYS_CREDIT_ENDDATE", "bureau")
before = bureau["DAYS_CREDIT_ENDDATE"].copy()
fig, ax = plt.subplots(figsize=(8, 4))
plot_numeric(before, "DAYS_CREDIT_ENDDATE before", bins=60, ax=ax)
plt.tight_layout()
plt.show()

bureau["DAYS_CREDIT_ENDDATE_MISSING"] = bureau["DAYS_CREDIT_ENDDATE"].isna().astype("int8")
print(">0 (after app):", int((bureau["DAYS_CREDIT_ENDDATE"] > 0).sum()))
print("<0 (before app):", int((bureau["DAYS_CREDIT_ENDDATE"] < 0).sum()))

record(
    "bureau",
    "DAYS_CREDIT_ENDDATE",
    "missing + future end dates (>0)",
    "keep NaN + missing flag; for PD features prefer closed/past info only",
)

## Feature: `DAYS_ENDDATE_FACT`

In [ ]:
feature_report(bureau, "DAYS_ENDDATE_FACT", "bureau")
fig, ax = plt.subplots(figsize=(8, 4))
plot_numeric(bureau["DAYS_ENDDATE_FACT"], "DAYS_ENDDATE_FACT", bins=60, ax=ax)
plt.tight_layout()
plt.show()
bureau["DAYS_ENDDATE_FACT_MISSING"] = bureau["DAYS_ENDDATE_FACT"].isna().astype("int8")
record(
    "bureau",
    "DAYS_ENDDATE_FACT",
    "missing when still active",
    "keep NaN + missing flag",
)

## Feature: `AMT_CREDIT_MAX_OVERDUE`

In [ ]:
feature_report(bureau, "AMT_CREDIT_MAX_OVERDUE", "bureau")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_numeric(
    np.log1p(bureau["AMT_CREDIT_MAX_OVERDUE"]),
    "log1p(AMT_CREDIT_MAX_OVERDUE)",
    ax=axes[0],
)
sns.countplot(x=bureau["AMT_CREDIT_MAX_OVERDUE"].isna().astype(int), ax=axes[1], color="salmon")
axes[1].set_title("missing flag")
plt.tight_layout()
plt.show()
bureau["AMT_CREDIT_MAX_OVERDUE_MISSING"] = bureau["AMT_CREDIT_MAX_OVERDUE"].isna().astype("int8")
record(
    "bureau",
    "AMT_CREDIT_MAX_OVERDUE",
    "high missing",
    "keep NaN + missing flag",
)

## Features: `AMT_CREDIT_SUM`, `AMT_CREDIT_SUM_DEBT`, `AMT_CREDIT_SUM_LIMIT`, `AMT_CREDIT_SUM_OVERDUE`

In [ ]:
amt_cols = [
    "AMT_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_LIMIT",
    "AMT_CREDIT_SUM_OVERDUE",
]
for col in amt_cols:
    feature_report(bureau, col, "bureau")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_numeric(np.log1p(bureau[col].clip(lower=0)), f"log1p({col})", ax=axes[0])
    sns.countplot(x=bureau[col].isna().astype(int), ax=axes[1], color="salmon")
    axes[1].set_title(f"{col} missing")
    plt.tight_layout()
    plt.show()
    if bureau[col].isna().any():
        bureau[f"{col}_MISSING"] = bureau[col].isna().astype("int8")
        record("bureau", col, "missing values", f"keep NaN + {col}_MISSING")
    else:
        record("bureau", col, "none", "keep as-is")

## Feature: `CNT_CREDIT_PROLONG`

In [ ]:
feature_report(bureau, "CNT_CREDIT_PROLONG", "bureau")
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(x=bureau["CNT_CREDIT_PROLONG"].clip(upper=5), ax=ax, color="steelblue")
ax.set_title("CNT_CREDIT_PROLONG (clipped at 5 for plot)")
plt.tight_layout()
plt.show()
record("bureau", "CNT_CREDIT_PROLONG", "none", "keep as-is")

## Feature: `CREDIT_TYPE`

In [ ]:
feature_report(bureau, "CREDIT_TYPE", "bureau")
plot_categorical(bureau["CREDIT_TYPE"], "bureau.CREDIT_TYPE", top_n=15, figsize=(8, 6))
record("bureau", "CREDIT_TYPE", "none", "keep as-is")

## Feature: `DAYS_CREDIT_UPDATE`

A few positive values exist; treat as dirty / rare.

In [ ]:
feature_report(bureau, "DAYS_CREDIT_UPDATE", "bureau")
n_pos = int((bureau["DAYS_CREDIT_UPDATE"] > 0).sum())
print(">0 rows:", n_pos)

before = bureau["DAYS_CREDIT_UPDATE"].copy()
bureau.loc[bureau["DAYS_CREDIT_UPDATE"] > 0, "DAYS_CREDIT_UPDATE"] = np.nan
bureau["DAYS_CREDIT_UPDATE_POS_FLAG"] = (before > 0).astype("int8")

before_after_numeric(
    before,
    bureau["DAYS_CREDIT_UPDATE"],
    "before: DAYS_CREDIT_UPDATE",
    "after: positive -> NaN",
    bins=60,
)
record(
    "bureau",
    "DAYS_CREDIT_UPDATE",
    f"{n_pos} positive values",
    "set >0 to NaN + DAYS_CREDIT_UPDATE_POS_FLAG",
)

## Feature: `AMT_ANNUITY` (bureau)

In [ ]:
feature_report(bureau, "AMT_ANNUITY", "bureau")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_numeric(np.log1p(bureau["AMT_ANNUITY"]), "log1p(AMT_ANNUITY)", ax=axes[0])
sns.countplot(x=bureau["AMT_ANNUITY"].isna().astype(int), ax=axes[1], color="salmon")
axes[1].set_title("AMT_ANNUITY missing")
plt.tight_layout()
plt.show()
bureau["AMT_ANNUITY_MISSING"] = bureau["AMT_ANNUITY"].isna().astype("int8")
record(
    "bureau",
    "AMT_ANNUITY",
    "very high missing (~70%+)",
    "keep NaN + missing flag; use cautiously in aggregates",
)

In [ ]:
bureau_out = PROCESSED_DATA_DIR / "bureau_clean.parquet"
bureau.to_parquet(bureau_out, index=False)
print("wrote", bureau_out, "shape", bureau.shape)
del bureau
gc.collect()

---
# B. `bureau_balance.csv`

Monthly bureau status. Large file -- sample for plots, chunked counts for full-data stats.

In [ ]:
bb_path = RAW_DATA_DIR / "bureau_balance.csv"
print("file_mb:", round(bb_path.stat().st_size / 1024**2, 1))

status_counts = chunked_value_counts(bb_path, "STATUS", top_n=20)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=status_counts.values, y=status_counts.index.astype(str), ax=ax, color="teal")
ax.set_title("bureau_balance.STATUS (full data, chunked counts)")
plt.tight_layout()
plt.show()
display(status_counts)

bb_sample = sample_csv(bb_path, n=SAMPLE_N)
feature_report(bb_sample, "MONTHS_BALANCE", "bureau_balance(sample)")
print("MONTHS_BALANCE > 0 in sample:", int((bb_sample["MONTHS_BALANCE"] > 0).sum()))
fig, ax = plt.subplots(figsize=(8, 4))
plot_numeric(bb_sample["MONTHS_BALANCE"], "MONTHS_BALANCE sample", bins=60, ax=ax)
plt.tight_layout()
plt.show()

record("bureau_balance", "STATUS", "none", "keep; C=closed, X=unknown, 0-5 DPD buckets")
record(
    "bureau_balance",
    "MONTHS_BALANCE",
    "relative month offset",
    "keep; use only MONTHS_BALANCE <= 0 at feature time",
)
record(
    "bureau_balance",
    "table",
    "very large",
    "no full rewrite; cleaning rules applied in DuckDB SQL later",
)

del bb_sample
gc.collect()

---
# C. `previous_application.csv`

Prior Home Credit applications. Day fields often use sentinel `365243`.

In [ ]:
prev_path = RAW_DATA_DIR / "previous_application.csv"
prev = pd.read_csv(prev_path, low_memory=False)
print("shape:", prev.shape)

miss = (prev.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
fig, ax = plt.subplots(figsize=(8, 8))
sns.barplot(x=miss.values, y=miss.index, ax=ax, color="salmon")
ax.set_title("previous_application missingness (>0)")
ax.set_xlabel("% missing")
plt.tight_layout()
plt.show()

## Feature: `NAME_CONTRACT_STATUS`

In [ ]:
feature_report(prev, "NAME_CONTRACT_STATUS", "previous_application")
plot_categorical(prev["NAME_CONTRACT_STATUS"], "previous_application.NAME_CONTRACT_STATUS")
record("previous_application", "NAME_CONTRACT_STATUS", "none", "keep; core for approval/refusal rates")

## Feature: `DAYS_DECISION`

In [ ]:
feature_report(prev, "DAYS_DECISION", "previous_application")
print(">0:", int((prev["DAYS_DECISION"] > 0).sum()))
print("sentinel:", int((prev["DAYS_DECISION"] == DAY_SENTINEL).sum()))
fig, ax = plt.subplots(figsize=(8, 4))
plot_numeric(prev["DAYS_DECISION"], "DAYS_DECISION", bins=60, ax=ax)
plt.tight_layout()
plt.show()
record(
    "previous_application",
    "DAYS_DECISION",
    "relative offset",
    "keep; filter DAYS_DECISION <= 0 for application-time features",
)

## Features: day sentinels `DAYS_FIRST_DRAWING`, `DAYS_FIRST_DUE`, `DAYS_LAST_DUE*`, `DAYS_TERMINATION`

In [ ]:
day_sentinel_cols = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
]

for col in day_sentinel_cols:
    feature_report(prev, col, "previous_application")
    before = prev[col].copy()
    n_sent = int((before == DAY_SENTINEL).sum())
    print(f"sentinel {DAY_SENTINEL}: {n_sent} ({100*n_sent/len(prev):.2f}%)")

    cleaned, flag = replace_day_sentinel(before)
    prev[col] = cleaned
    prev[f"{col}_ANOMALY_FLAG"] = flag

    before_after_numeric(
        before,
        prev[col],
        f"before: {col}",
        f"after: {col} (sentinel -> NaN)",
        bins=60,
    )
    record(
        "previous_application",
        col,
        f"sentinel {DAY_SENTINEL}",
        f"set sentinel to NaN + {col}_ANOMALY_FLAG",
        f"n={n_sent}",
    )

## Features: amounts `AMT_APPLICATION`, `AMT_CREDIT`, `AMT_ANNUITY`, `AMT_DOWN_PAYMENT`, `AMT_GOODS_PRICE`

In [ ]:
prev_amt_cols = [
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
]
for col in prev_amt_cols:
    feature_report(prev, col, "previous_application")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_numeric(np.log1p(prev[col].clip(lower=0)), f"log1p({col})", ax=axes[0])
    sns.countplot(x=prev[col].isna().astype(int), ax=axes[1], color="salmon")
    axes[1].set_title(f"{col} missing")
    plt.tight_layout()
    plt.show()
    if prev[col].isna().any():
        prev[f"{col}_MISSING"] = prev[col].isna().astype("int8")
        record("previous_application", col, "missing", f"keep NaN + {col}_MISSING")
    else:
        record("previous_application", col, "none", "keep as-is")

## Features: categoricals with XNA / XAP placeholders

In [ ]:
cat_cols = [
    "NAME_CONTRACT_TYPE",
    "NAME_CASH_LOAN_PURPOSE",
    "CODE_REJECT_REASON",
    "NAME_CLIENT_TYPE",
    "NAME_GOODS_CATEGORY",
    "NAME_PORTFOLIO",
    "NAME_PRODUCT_TYPE",
    "CHANNEL_TYPE",
    "NAME_SELLER_INDUSTRY",
    "NAME_YIELD_GROUP",
    "PRODUCT_COMBINATION",
    "NAME_PAYMENT_TYPE",
    "NAME_TYPE_SUITE",
]

for col in cat_cols:
    if col not in prev.columns:
        continue
    feature_report(prev, col, "previous_application")
    plot_categorical(prev[col], f"before: {col}", top_n=15)

    before_n = prev[col].copy()
    n_x = int(before_n.isin(["XNA", "XAP"]).sum())
    n_miss = int(before_n.isna().sum())
    prev[col] = (
        before_n.replace({"XNA": "Missing", "XAP": "Missing"})
        .fillna("Missing")
    )
    plot_categorical(prev[col], f"after: {col} (XNA/XAP/NaN -> Missing)", top_n=15)
    record(
        "previous_application",
        col,
        f"XNA/XAP/NaN n_x={n_x} n_miss={n_miss}",
        "map XNA/XAP/NaN -> Missing",
    )

## Features: `RATE_DOWN_PAYMENT`, `RATE_INTEREST_*`, `CNT_PAYMENT`, `SELLERPLACE_AREA`

In [ ]:
extra_num = [
    "RATE_DOWN_PAYMENT",
    "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED",
    "CNT_PAYMENT",
    "SELLERPLACE_AREA",
]
for col in extra_num:
    feature_report(prev, col, "previous_application")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_numeric(prev[col], col, ax=axes[0])
    sns.countplot(x=prev[col].isna().astype(int), ax=axes[1], color="salmon")
    axes[1].set_title(f"{col} missing")
    plt.tight_layout()
    plt.show()
    if prev[col].isna().any():
        prev[f"{col}_MISSING"] = prev[col].isna().astype("int8")
        record("previous_application", col, "missing", f"keep NaN + {col}_MISSING")
    else:
        record("previous_application", col, "none", "keep as-is")

# SELLERPLACE_AREA uses -1 as unknown in Home Credit
if "SELLERPLACE_AREA" in prev.columns:
    n_m1 = int((prev["SELLERPLACE_AREA"] == -1).sum())
    print("SELLERPLACE_AREA == -1:", n_m1)
    before = prev["SELLERPLACE_AREA"].copy()
    prev.loc[prev["SELLERPLACE_AREA"] == -1, "SELLERPLACE_AREA"] = np.nan
    prev["SELLERPLACE_AREA_MISSING"] = prev["SELLERPLACE_AREA"].isna().astype("int8")
    before_after_numeric(
        before,
        prev["SELLERPLACE_AREA"],
        "before: SELLERPLACE_AREA",
        "after: -1 -> NaN",
    )
    record(
        "previous_application",
        "SELLERPLACE_AREA",
        f"-1 unknown n={n_m1}",
        "set -1 to NaN + missing flag",
    )

In [ ]:
prev_out = PROCESSED_DATA_DIR / "previous_application_clean.parquet"
prev.to_parquet(prev_out, index=False)
print("wrote", prev_out, "shape", prev.shape)
del prev
gc.collect()

---
# D. `installments_payments.csv`

Repayment behaviour (closest analogue to mobile-money repayment history).
Large file: sample plots + chunked cleaning rules / stats.

In [ ]:
inst_path = RAW_DATA_DIR / "installments_payments.csv"
print("file_mb:", round(inst_path.stat().st_size / 1024**2, 1))

for col in ["DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"]:
    stats = chunked_null_and_sentinel(inst_path, col)
    print(col, stats)

inst = sample_csv(inst_path, n=SAMPLE_N)
print("sample shape:", inst.shape)

# DAYS_ENTRY_PAYMENT missing => payment not recorded / unpaid timing unknown
feature_report(inst, "DAYS_ENTRY_PAYMENT", "installments(sample)")
feature_report(inst, "DAYS_INSTALMENT", "installments(sample)")

inst["DAYS_LATE"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_numeric(inst["DAYS_LATE"], "DAYS_LATE sample (entry - instalment)", bins=60, ax=axes[0])
sns.countplot(x=inst["DAYS_ENTRY_PAYMENT"].isna().astype(int), ax=axes[1], color="salmon")
axes[1].set_title("DAYS_ENTRY_PAYMENT missing in sample")
plt.tight_layout()
plt.show()

inst["PAYMENT_RATIO"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)
fig, ax = plt.subplots(figsize=(8, 4))
plot_numeric(
    inst["PAYMENT_RATIO"].clip(0, 3),
    "PAYMENT_RATIO sample clipped [0,3]",
    bins=40,
    ax=ax,
)
plt.tight_layout()
plt.show()

print("DAYS_INSTALMENT > 0 in sample:", int((inst["DAYS_INSTALMENT"] > 0).sum()))
print("DAYS_ENTRY_PAYMENT > 0 in sample:", int((inst["DAYS_ENTRY_PAYMENT"] > 0).sum()))

record(
    "installments_payments",
    "DAYS_ENTRY_PAYMENT",
    "missing when payment timing unknown",
    "keep NaN; derive late/underpay features only when both day fields present",
)
record(
    "installments_payments",
    "DAYS_INSTALMENT",
    "relative offset",
    "keep; use only offsets available at application time in SQL",
)
record(
    "installments_payments",
    "AMT_PAYMENT/AMT_INSTALMENT",
    "under/over payment",
    "derive payment ratio / late flags in feature SQL",
)
record(
    "installments_payments",
    "table",
    "very large",
    "no full parquet rewrite; cleaning applied in DuckDB SQL",
)

del inst
gc.collect()

---
# E. `POS_CASH_balance.csv`

In [ ]:
pos_path = RAW_DATA_DIR / "POS_CASH_balance.csv"
print("file_mb:", round(pos_path.stat().st_size / 1024**2, 1))

pos_status = chunked_value_counts(pos_path, "NAME_CONTRACT_STATUS", top_n=15)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=pos_status.values, y=pos_status.index.astype(str), ax=ax, color="teal")
ax.set_title("POS_CASH NAME_CONTRACT_STATUS (full, chunked)")
plt.tight_layout()
plt.show()

pos = sample_csv(pos_path, n=SAMPLE_N)
for col in ["MONTHS_BALANCE", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE", "SK_DPD", "SK_DPD_DEF"]:
    feature_report(pos, col, "POS_CASH(sample)")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    if col.startswith("SK_DPD"):
        plot_numeric(pos.loc[pos[col] > 0, col], f"{col} | >0 only (sample)", bins=40, ax=ax)
    else:
        plot_numeric(pos[col], f"{col} sample", bins=40, ax=ax)
    plt.tight_layout()
    plt.show()

print("MONTHS_BALANCE > 0:", int((pos["MONTHS_BALANCE"] > 0).sum()))

for col in ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]:
    stats = chunked_null_and_sentinel(pos_path, col)
    print(col, "null/sentinel stats", stats)

record("POS_CASH_balance", "MONTHS_BALANCE", "relative month", "use only <= 0")
record("POS_CASH_balance", "SK_DPD/SK_DPD_DEF", "delinquency days", "keep; aggregate in SQL")
record(
    "POS_CASH_balance",
    "CNT_INSTALMENT*",
    "some missing",
    "keep NaN; impute/aggregate carefully in SQL",
)
record("POS_CASH_balance", "table", "large", "rules in SQL; no full rewrite here")

del pos
gc.collect()

---
# F. `credit_card_balance.csv`

Note: source column `AMT_RECIVABLE` is a misspelling in the dataset; keep name for join compatibility.

In [ ]:
cc_path = RAW_DATA_DIR / "credit_card_balance.csv"
print("file_mb:", round(cc_path.stat().st_size / 1024**2, 1))

cc = sample_csv(cc_path, n=SAMPLE_N)
print("sample shape:", cc.shape)
miss = (cc.isna().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=miss.values, y=miss.index, ax=ax, color="salmon")
ax.set_title("credit_card_balance missingness (sample)")
plt.tight_layout()
plt.show()

for col in [
    "MONTHS_BALANCE",
    "AMT_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_CURRENT",
    "AMT_PAYMENT_TOTAL_CURRENT",
    "SK_DPD",
    "SK_DPD_DEF",
]:
    feature_report(cc, col, "credit_card(sample)")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    if col.startswith("AMT_"):
        plot_numeric(np.log1p(cc[col].clip(lower=0)), f"log1p({col}) sample", ax=ax)
    elif col.startswith("SK_DPD"):
        plot_numeric(cc.loc[cc[col] > 0, col], f"{col} >0 sample", ax=ax)
    else:
        plot_numeric(cc[col], f"{col} sample", ax=ax)
    plt.tight_layout()
    plt.show()

# Utilisation proxy on sample
cc["UTILISATION"] = cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
fig, ax = plt.subplots(figsize=(8, 4))
plot_numeric(cc["UTILISATION"].clip(-1, 2), "UTILISATION sample clipped [-1,2]", ax=ax)
plt.tight_layout()
plt.show()

cc_status = chunked_value_counts(cc_path, "NAME_CONTRACT_STATUS", top_n=10)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=cc_status.values, y=cc_status.index.astype(str), ax=ax, color="teal")
ax.set_title("credit_card NAME_CONTRACT_STATUS (full, chunked)")
plt.tight_layout()
plt.show()

record("credit_card_balance", "AMT_RECIVABLE", "dataset typo in name", "keep original column name")
record("credit_card_balance", "drawings/payment fields", "block missingness", "keep NaN; aggregate in SQL")
record("credit_card_balance", "MONTHS_BALANCE", "relative month", "use only <= 0")
record("credit_card_balance", "utilisation", "limit can be 0", "safe-divide in SQL")
record("credit_card_balance", "table", "large", "rules in SQL; no full rewrite here")

del cc
gc.collect()

---
# Save decision log

In [ ]:
out_decisions = PROCESSED_DATA_DIR / "child_tables_cleaning_decisions.json"
payload = {
    "day_sentinel": int(DAY_SENTINEL),
    "cleaned_parquets": [
        "bureau_clean.parquet",
        "previous_application_clean.parquet",
    ],
    "sql_only_tables": [
        "bureau_balance.csv",
        "installments_payments.csv",
        "POS_CASH_balance.csv",
        "credit_card_balance.csv",
    ],
    "decisions": DECISIONS,
}
out_decisions.write_text(json.dumps(payload, indent=2), encoding="ascii", errors="replace")

print("wrote", out_decisions)
print("n decisions:", len(DECISIONS))
decisions_df = pd.DataFrame(DECISIONS)
display(decisions_df.groupby("table").size().rename("n_decisions"))
display(decisions_df.tail(25))

## Summary

| Table | Handling |
|---|---|
| `bureau` | Cleaned parquet; positive `DAYS_CREDIT_UPDATE` nulled; missing flags |
| `previous_application` | Cleaned parquet; `365243` day sentinels nulled; XNA/XAP -> Missing |
| `bureau_balance` | Rules only (large); STATUS / MONTHS_BALANCE checked |
| `installments_payments` | Rules only; late/payment-ratio logic for SQL |
| `POS_CASH_balance` | Rules only; DPD + months balance |
| `credit_card_balance` | Rules only; utilisation safe-divide |

Outputs:
- `data/processed/bureau_clean.parquet`
- `data/processed/previous_application_clean.parquet`
- `data/processed/child_tables_cleaning_decisions.json`

Next (after this looks good): port Stage 2 into `src/data_quality.py` + report, then **switch to premium** for Stage 3 SQL features.